## Overview of AVFSpoof dataset

In [29]:
import torch

test = torch.load("../data/processed/asvspoof/planted_narrowband_0db/train.pt")

In [ ]:
test

dict_keys(['specs', 'clean_specs', 'labels', 'gt_masks', 'clip_ids'])

In [26]:
from pathlib import Path

from xai_bench.config import PrepareDataConfig
from xai_bench.data.artifacts import LogMelExtractor


cfg = PrepareDataConfig.from_yaml("../configs/prepare_data.yaml")
extractor = LogMelExtractor(cfg.audio)

data_root = Path("..")
raw_test_dir = data_root / cfg.paths.raw_audio_dir / "test"
processed_dir = data_root / cfg.paths.output_dir

In [27]:
import numpy as np
from IPython.display import Audio, display

from xai_bench.data.artifacts import inject_artifact, load_fixed_waveform

example_index = 0

paths = sorted(raw_test_dir.glob("*.flac"))
path = paths[example_index]

target_len = int(cfg.audio.clip_duration_sec * cfg.audio.sample_rate)
clean_wav = load_fixed_waveform(path, target_len)
planted_wav, gt_mask = inject_artifact(clean_wav, cfg.audio, cfg.artifact, extractor, np.random.default_rng(42))

print("Original:")
display(Audio(clean_wav.numpy(), rate=cfg.audio.sample_rate, normalize=False))
print("With artifact:")
display(Audio(planted_wav.numpy(), rate=cfg.audio.sample_rate, normalize=False))

Original:


With artifact:


In [ ]:
from xai_bench.plots import plot_example

paths = sorted(raw_test_dir.glob("*.flac"))
rng = np.random.default_rng(42)

for path in paths[:5]:
    clean_wav = load_fixed_waveform(path, target_len)
    planted_wav, gt_mask = inject_artifact(clean_wav, cfg.audio, cfg.artifact, extractor, rng)

    print(path.stem)
    plot_example(
        extractor(clean_wav),
        extractor(planted_wav),
        cfg.audio,
        gt_mask=gt_mask,
    )